In [1]:
# Chess Engine with PyTorch - GPU Optimized (Fixed Hanging Issues)
# Converted from TensorFlow implementation with GPU optimizations

import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import pgn, Board
from tqdm import tqdm
import pickle
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.cuda.amp as amp  # Mixed precision training

# ============================================================
# Data Loading Functions
# ============================================================

def load_pgn(file_path):
    """Load chess games from PGN file"""
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

def board_to_matrix(board: Board):
    """Convert chess board to 8x8x12 tensor representation"""
    matrix = np.zeros((8, 8, 12), dtype=np.float32)
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

def create_input_for_nn(games):
    """Create training data from chess games"""
    X = []
    y = []
    print("Processing games into training data...")
    for game in tqdm(games):
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

def encode_moves(moves):
    """Encode moves to integer labels"""
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

# ============================================================
# PyTorch Dataset - Memory Efficient Version
# ============================================================

class ChessDataset(Dataset):
    """Memory-efficient Custom Dataset for chess positions and moves"""
    def __init__(self, X, y):
        # Store as numpy arrays, convert to tensors on-the-fly in __getitem__
        print("Initializing dataset...")
        self.X = np.array(X, dtype=np.float32)
        self.y = np.array(y, dtype=np.int64)
        print(f"Dataset initialized with {len(self.X)} samples")
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Convert to tensor and transpose on-the-fly
        x = torch.from_numpy(self.X[idx]).permute(2, 0, 1)  # (H,W,C) -> (C,H,W)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y

# ============================================================
# Optimized PyTorch Model
# ============================================================

class ChessNet(nn.Module):
    """GPU-Optimized CNN model for chess move prediction"""
    def __init__(self, num_classes, dropout=0.3):
        super(ChessNet, self).__init__()
        
        # Convolutional layers with batch normalization
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=0)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.bn2 = nn.BatchNorm2d(128)
        
        # Additional conv layer for better feature extraction
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        # Global average pooling to reduce parameters
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Fully connected layers with dropout
        self.fc1 = nn.Linear(256, 512)
        self.dropout1 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(dropout)
        
        self.fc3 = nn.Linear(256, num_classes)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Conv block 1
        x = self.relu(self.bn1(self.conv1(x)))
        
        # Conv block 2
        x = self.relu(self.bn2(self.conv2(x)))
        
        # Conv block 3
        x = self.relu(self.bn3(self.conv3(x)))
        
        # Global average pooling
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.dropout1(self.relu(self.fc1(x)))
        x = self.dropout2(self.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

# ============================================================
# GPU-Optimized Training Function
# ============================================================

def train_model(model, train_loader, val_loader, epochs, device, lr=0.001, use_amp=True):
    """Train the chess model with GPU optimizations"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    
    # Learning rate scheduler for better convergence
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    # Mixed precision training scaler (faster on GPU)
    scaler = amp.GradScaler() if use_amp and torch.cuda.is_available() else None
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for batch_idx, (inputs, labels) in enumerate(train_pbar):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Mixed precision training
            if use_amp and scaler is not None:
                with amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            # Update progress bar every 10 batches to reduce overhead
            if batch_idx % 10 == 0:
                train_pbar.set_postfix({
                    'loss': f'{train_loss/(batch_idx+1):.4f}',
                    'acc': f'{100*train_correct/train_total:.2f}%'
                })
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                if use_amp and scaler is not None:
                    with amp.autocast():
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * val_correct / val_total
        
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        # Learning rate scheduling
        scheduler.step(epoch_val_loss)
        
        print(f'Epoch {epoch+1}/{epochs} - '
              f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% - '
              f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')
        
        # Save best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            os.makedirs("simulated_filtered_model_pytorch", exist_ok=True)
            torch.save(model.state_dict(), "simulated_filtered_model_pytorch/best_model.pth")
            print(f"  → Saved best model (val_loss: {best_val_loss:.4f})")
    
    return history

# ============================================================
# Main Training Script
# ============================================================

if __name__ == "__main__":
    print("="*60)
    print("Chess Engine Training - PyTorch")
    print("="*60)
    
    # ========================================
    # GPU SELECTION - CHANGE THIS VALUE
    # ========================================
    # For DirectML (AMD): 0, 1, 2, etc. (GPU index)
    # For CUDA (NVIDIA): 0, 1, 2, etc. (GPU index)
    # Set to None for auto-detection
    GPU_INDEX = 1  # <-- CHANGE THIS TO SELECT DIFFERENT GPU
    # ========================================
    
    # Set device - Try DirectML first (AMD), then CUDA (NVIDIA), then CPU
    device = None
    device_type = "cpu"
    
    # Try DirectML (for AMD GPUs on Windows)
    try:
        import torch_directml
        
        # List available DirectML devices
        print("\n" + "="*60)
        print("Available DirectML Devices:")
        print("="*60)
        device_count = torch_directml.device_count()
        for i in range(device_count):
            print(f"  Device {i}: {torch_directml.device_name(i)}")
        
        # Select GPU
        if GPU_INDEX is not None and GPU_INDEX < device_count:
            device = torch_directml.device(GPU_INDEX)
            print(f"\n✓ Using device: DirectML Device {GPU_INDEX}")
            print(f"  GPU: {torch_directml.device_name(GPU_INDEX)}")
        else:
            device = torch_directml.device()  # Use default
            print(f"\n✓ Using device: DirectML (Default GPU)")
        
        device_type = "directml"
        
    except ImportError:
        print("\n✗ torch-directml not found")
    except Exception as e:
        print(f"\n✗ DirectML error: {e}")
    
    # Try CUDA (for NVIDIA GPUs)
    if device is None and torch.cuda.is_available():
        print("\n" + "="*60)
        print("Available CUDA Devices:")
        print("="*60)
        device_count = torch.cuda.device_count()
        for i in range(device_count):
            print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
        
        # Select GPU
        if GPU_INDEX is not None and GPU_INDEX < device_count:
            device = torch.device(f"cuda:{GPU_INDEX}")
            print(f"\n✓ Using device: CUDA Device {GPU_INDEX}")
            print(f"  GPU: {torch.cuda.get_device_name(GPU_INDEX)}")
        else:
            device = torch.device("cuda")
            print(f"\n✓ Using device: CUDA (Default GPU)")
            print(f"  GPU: {torch.cuda.get_device_name(0)}")
        
        device_type = "cuda"
        print(f"CUDA Version: {torch.version.cuda}")
        if GPU_INDEX is not None:
            print(f"Available GPU memory: {torch.cuda.get_device_properties(GPU_INDEX).total_memory / 1e9:.2f} GB")
        else:
            print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        torch.backends.cudnn.benchmark = True
    
    # Fallback to CPU
    if device is None:
        device = torch.device("cpu")
        device_type = "cpu"
        print(f"\n⚠ Using device: CPU (No GPU detected)")
        print("For AMD GPU support, install: pip install torch-directml")
    
    # Set number of threads for CPU operations
    torch.set_num_threads(4)
    
    # Load game files
    print("\n" + "="*60)
    print("Step 1: Loading Game Files")
    print("="*60)
    files = [file for file in os.listdir("simulated_games_filtered_PGN") 
             if file.endswith(".pgn")]
    print(f"Found {len(files)} PGN files")
    
    games = []
    for file in tqdm(files, desc="Loading PGN files"):
        games.extend(load_pgn(f"simulated_games_filtered_PGN/{file}"))
    
    print(f"✓ Loaded {len(games)} games")
    
    # Create training data
    print("\n" + "="*60)
    print("Step 2: Creating Training Data")
    print("="*60)
    X, y = create_input_for_nn(games)
    y_encoded, move_to_int = encode_moves(y)
    
    print(f"✓ Training samples: {len(X):,}")
    print(f"✓ Unique moves: {len(move_to_int):,}")
    
    # Create dataset
    print("\n" + "="*60)
    print("Step 3: Preparing Dataset")
    print("="*60)
    dataset = ChessDataset(X, y_encoded)
    
    # Split into train and validation
    val_size = int(0.1 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    print(f"✓ Train samples: {train_size:,}")
    print(f"✓ Validation samples: {val_size:,}")
    
    # Create data loaders
    batch_size = 256
    num_workers = 0  # Safe for Windows
    
    print("\nCreating data loaders...")
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=(device_type == "cuda")  # Only for CUDA
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=(device_type == "cuda")  # Only for CUDA
    )
    print(f"✓ Batch size: {batch_size}")
    print(f"✓ Training batches: {len(train_loader)}")
    print(f"✓ Validation batches: {len(val_loader)}")
    
    # Create model
    print("\n" + "="*60)
    print("Step 4: Building Model")
    print("="*60)
    model = ChessNet(num_classes=len(move_to_int)).to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Total parameters: {total_params:,}")
    print(f"✓ Trainable parameters: {trainable_params:,}")
    
    # Train model
    print("\n" + "="*60)
    print("Step 5: Training Model")
    print("="*60)
    print(f"Epochs: 50")
    print(f"Mixed precision: {'Enabled' if device_type in ['cuda', 'directml'] else 'Disabled (CPU)'}")
    print()
    
    history = train_model(
        model, 
        train_loader, 
        val_loader, 
        epochs=50, 
        device=device,
        lr=0.001,
        use_amp=(device_type == "cuda")  # Only use AMP with CUDA
    )
    
    # Save model
    print("\n" + "="*60)
    print("Step 6: Saving Model and Artifacts")
    print("="*60)
    os.makedirs("simulated_filtered_model_pytorch", exist_ok=True)
    
    torch.save(model.state_dict(), "simulated_filtered_model_pytorch/chess_model_50epochs.pth")
    print("✓ Saved final model")
    
    # Save move encodings
    with open("simulated_filtered_model_pytorch/move_to_int.pkl", "wb") as f:
        pickle.dump(move_to_int, f)
    
    int_to_move = {v: k for k, v in move_to_int.items()}
    with open("simulated_filtered_model_pytorch/int_to_move.pkl", "wb") as f:
        pickle.dump(int_to_move, f)
    print("✓ Saved move encodings")
    
    # Save configuration
    config = {
        "epochs": 50,
        "batch_size": batch_size,
        "validation_split": 0.1,
        "optimizer": "AdamW",
        "learning_rate": 0.001,
        "input_shape": [12, 8, 8],
        "num_classes": len(move_to_int),
        "mixed_precision": (device_type == "cuda"),
        "num_workers": num_workers,
        "device": device_type
    }
    
    with open("simulated_filtered_model_pytorch/train_config.json", "w") as f:
        json.dump(config, f, indent=4)
    print("✓ Saved configuration")
    
    # Save training history
    with open("simulated_filtered_model_pytorch/training_history.pkl", "wb") as f:
        pickle.dump(history, f)
    print("✓ Saved training history")
    
    print("\n" + "="*60)
    print("Training Complete!")
    print("="*60)
    print(f"Final model: simulated_filtered_model_pytorch/chess_model_50epochs.pth")
    print(f"Best model: simulated_filtered_model_pytorch/best_model.pth")
    print(f"Final train accuracy: {history['train_acc'][-1]:.2f}%")
    print(f"Final val accuracy: {history['val_acc'][-1]:.2f}%")

Chess Engine Training - PyTorch

Available DirectML Devices:
  Device 0: AMD Radeon(TM) Graphics 
  Device 1: AMD Radeon RX 7800 XT 

✓ Using device: DirectML Device 1
  GPU: AMD Radeon RX 7800 XT 

Step 1: Loading Game Files
Found 11 PGN files


Loading PGN files: 100%|██████████| 11/11 [00:26<00:00,  2.39s/it]


✓ Loaded 10085 games

Step 2: Creating Training Data
Processing games into training data...


100%|██████████| 10085/10085 [00:32<00:00, 310.01it/s]


✓ Training samples: 1,119,457
✓ Unique moves: 1,945

Step 3: Preparing Dataset
Initializing dataset...
Dataset initialized with 1119457 samples
✓ Train samples: 1,007,512
✓ Validation samples: 111,945

Creating data loaders...
✓ Batch size: 256
✓ Training batches: 3936
✓ Validation batches: 438

Step 4: Building Model
✓ Total parameters: 1,139,673
✓ Trainable parameters: 1,139,673

Step 5: Training Model
Epochs: 50
Mixed precision: Enabled



...

Epoch 48/50 - Train Loss: 4.7117, Train Acc: 9.56% - Val Loss: 4.7328, Val Acc: 9.21%
  → Saved best model (val_loss: 4.7328)


Epoch 49/50 [Train]: 100%|██████████| 3936/3936 [00:54<00:00, 71.86it/s, loss=4.7044, acc=9.57%]


Epoch 49/50 - Train Loss: 4.7045, Train Acc: 9.57% - Val Loss: 4.7336, Val Acc: 9.19%


Epoch 50/50 [Train]: 100%|██████████| 3936/3936 [00:51<00:00, 76.36it/s, loss=4.7026, acc=9.59%]


Epoch 50/50 - Train Loss: 4.7026, Train Acc: 9.59% - Val Loss: 4.7307, Val Acc: 9.32%
  → Saved best model (val_loss: 4.7307)

Step 6: Saving Model and Artifacts
✓ Saved final model
✓ Saved move encodings
✓ Saved configuration
✓ Saved training history

Training Complete!
Final model: simulated_filtered_model_pytorch/chess_model_50epochs.pth
Best model: simulated_filtered_model_pytorch/best_model.pth
Final train accuracy: 9.59%
Final val accuracy: 9.32%
